# 01 - Exploración inicial del dataset ambiental

Este cuaderno revisa la calidad del archivo de calidad del aire de la OMS, identifica columnas clave, valores faltantes y patrones exploratorios para preparar el análisis predictivo.

In [ ]:
# Instalación de dependencias si es necesario
# %pip install pandas numpy matplotlib seaborn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/who_ambient_air_quality_database_version_2024_(v6.1).xlsx")
SHEET = "Update 2024 (V6.1)"

# Cargar datos
raw_df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
print(f"Filas: {raw_df.shape[0]} | Columnas: {raw_df.shape[1]}")
raw_df.head()

In [ ]:
raw_df.info()
print("\nValores faltantes por columna:")
print(raw_df.isna().sum().sort_values(ascending=False).head(15))
print(f"\nDuplicados: {raw_df.duplicated().sum()}")

In [ ]:
# Convertir variables numéricas relevantes
for col in ["year", "population", "latitude", "longitude", "pm10_concentration", "pm25_concentration", "no2_concentration", "pm10_tempcov", "pm25_tempcov", "no2_tempcov", "who_ms"]:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

numeric_cols = ["year", "population", "latitude", "longitude", "pm10_concentration", "pm25_concentration", "no2_concentration", "pm10_tempcov", "pm25_tempcov", "no2_tempcov"]
raw_df[numeric_cols].describe().T

In [ ]:
# Distribuciones de concentraciones
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["pm10_concentration", "pm25_concentration", "no2_concentration"]):
    sns.histplot(raw_df[col].dropna(), bins=40, kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Tendencia por año
trend = raw_df.groupby("year")["pm25_concentration"].mean().reset_index()
trend = trend.dropna()

plt.figure(figsize=(10, 4))
sns.lineplot(data=trend, x="year", y="pm25_concentration")
plt.title("Promedio de PM2.5 por año")
plt.xlabel("Año")
plt.ylabel("PM2.5 promedio")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Relación entre PM2.5 y NO2
plt.figure(figsize=(7, 4))
sns.scatterplot(data=raw_df, x="no2_concentration", y="pm25_concentration", alpha=0.4)
plt.title("Relación PM2.5 vs NO2")
plt.xlabel("NO2")
plt.ylabel("PM2.5")
plt.show()

In [ ]:
# Resumen por región
region_summary = raw_df.groupby("who_region")["pm25_concentration"].agg(["mean", "median", "std", "count"]).reset_index()
region_summary.sort_values("mean", ascending=False)

In [ ]:
# Correlación numérica
plt.figure(figsize=(10, 7))
correlation_matrix = raw_df[numeric_cols].corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=False, cmap="coolwarm")
plt.title("Mapa de correlación")
plt.show()